# AI Invoice Fine-Tuning Notebook (Free Colab Workflow)

This version follows the safer free workflow:

**Train in Colab → Export merged model directly to Google Drive → Download from Drive → Convert to GGUF locally on Windows**

Do **not** run `llama.cpp` conversion inside Colab. That is the part that usually causes disk-space failures.

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd /content/LLaMA-Factory
!pip install -e .[torch,metrics]

In [ ]:
!llamafactory-cli train colab_llamafactory_config.yaml

## Quick verification in Colab

This loads the base Qwen model plus the LoRA adapter saved by LLaMA-Factory, so you can confirm the fine-tuned behavior before exporting.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/LLaMA-Factory/saves/qwen2.5-1.5b/lora/sft"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

prompt = r"""You are a structured invoice data extraction assistant.
Extract invoice data from the input text and return ONLY valid JSON matching the schema.

plz issue bill for Vertex Marketting, emal vertex@mail.com, addres 39 main road.
3 Photography Package AED 6334, 1 Monitor Arm AED 3986, 1 Conference Table AED 2408.
disc 0% and tax 5%. due 2026-09-12. note soft copy.
"""

messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Mount Google Drive

Use Drive as the export destination so Colab does not run out of local disk space during merge/export.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Export the merged model directly to Drive

This creates a standalone Hugging Face model folder in Drive. You can later download this folder to your Windows machine and convert it to GGUF locally.

In [ ]:
!llamafactory-cli export   --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct   --adapter_name_or_path /content/LLaMA-Factory/saves/qwen2.5-1.5b/lora/sft   --template qwen   --finetuning_type lora   --export_dir /content/drive/MyDrive/invoice-models/invoice-qwen2.5-1.5b-merged   --export_size 2   --export_device cpu

In [ ]:
!ls -lh /content/drive/MyDrive/invoice-models/invoice-qwen2.5-1.5b-merged

## Optional cleanup after export

Run this only after the Drive export is complete. It frees space in the Colab runtime.

Do **not** delete the Drive folder.

In [ ]:
!rm -rf /root/.cache/huggingface
!rm -rf /content/llama.cpp

## After Colab

1. Open Google Drive and download the folder:

   `/MyDrive/invoice-models/invoice-qwen2.5-1.5b-merged`

2. On Windows, convert it to GGUF with local `llama.cpp`.

3. Run `llama-server.exe` with the GGUF model.

This notebook intentionally stops before GGUF conversion so it stays free-tier friendly.